# OpenAlex Local - Quickstart Guide

This notebook demonstrates the core features of `openalex-local`:

1. **Search** - Full-text search across 459M+ scholarly works
2. **Get** - Retrieve works by DOI or OpenAlex ID
3. **Citations** - Generate APA and BibTeX citations
4. **Cache** - Local caching for offline analysis
5. **Async** - Concurrent search operations

In [1]:
# Import openalex-local
import openalex_local as oal
from openalex_local import search, get, cache, aio, enrich_ids

## 1. Basic Search

Search across titles and abstracts using FTS5 syntax.

In [2]:
# Simple search
results = search("machine learning neural networks", limit=5)

print(f"Found {results.total:,} matches in {results.elapsed_ms:.1f}ms\n")

for i, work in enumerate(results.works, 1):
    print(f"{i}. {work.title} ({work.year})")
    print(f"   DOI: {work.doi or 'N/A'}")
    print()

Found 14 matches in 3.1ms

1. Data Mining: Practical Machine Learning Tools and Techniques (2011)
   DOI: 10.1016/c2009-0-19715-5

2. TensorFlow: A system for large-scale machine learning (2016)
   DOI: 10.48550/arxiv.1605.08695

3. TensorFlow: Large-Scale Machine Learning on Heterogeneous Distributed Systems (2016)
   DOI: 10.48550/arxiv.1603.04467

4. TensorFlow: a system for large-scale machine learning (2016)
   DOI: 10.5555/3026877.3026899

5. Practical Bayesian Optimization of Machine Learning Algorithms (2012)
   DOI: 10.48550/arxiv.1206.2944



## 2. Get by DOI

Retrieve a specific work with full metadata.

In [3]:
# Get a work by its identifier.
# Pick a concrete id from a search so this cell works against any DB.
hit = search("machine learning", limit=1).works[0]
work = get(hit.openalex_id)

if work:
    print(f"Title: {work.title}")
    print(f"Year: {work.year}")
    print(f"Citations: {work.cited_by_count:,}")
    if work.authors:
        print(f"Authors: {', '.join(work.authors[:3])}")
    if work.abstract:
        print(f"\nAbstract preview: {work.abstract[:200]}...")

Title: Scikit-learn: Machine Learning in Python
Year: 2012
Citations: 63,735
Authors: [, ", F

Abstract preview: Scikit-learn is a Python module integrating a wide range of state-of-the-art machine learning algorithms for medium-scale supervised and unsupervised problems. This package focuses on bringing machine...


## 3. Citations

Generate formatted citations in APA or BibTeX style.

In [4]:
# APA citation
print("APA Citation:")
print(work.citation("apa"))
print()

# BibTeX entry
print("BibTeX Entry:")
print(work.citation("bibtex"))

APA Citation:
[, ", F, a, b, i, \, u, 0, 0, e, 1, n, , P, e, d, r, e, ..., & ] (2012) Scikit-learn: Machine Learning in Python. *arXiv (Cornell University)*. https://doi.org/10.48550/arxiv.1201.0490

BibTeX Entry:
@article{W2101234009,
  title = {Scikit-learn: Machine Learning in Python},
  author = {[ and " and F and a and b and i and \ and u and 0 and 0 and e and 1 and n and   and P and e and d and r and e and g and o and s and a and " and , and   and " and G and a and \ and u and 0 and 0 and e and b and l and   and V and a and r and o and q and u and a and u and x and " and , and   and " and A and l and e and x and a and n and d and r and e and   and G and r and a and m and f and o and r and t and " and , and   and " and V and i and n and c and e and n and t and   and M and i and c and h and e and l and " and , and   and " and B and e and r and t and r and a and n and d and   and T and h and i and r and i and o and n and " and , and   and " and O and l and i and v and i and e and r 

## 4. Cache Workflow

Create local caches for offline analysis.

In [5]:
# Create a cache from a search.
if cache.exists("demo_cache"):
    cache.delete("demo_cache")

info = cache.create("demo_cache", query="machine learning", limit=50)
print(f"Created cache with {info.count} papers")

# Get statistics
stats = cache.stats("demo_cache")
print(f"Year range: {stats['year_min']} - {stats['year_max']}")
print(f"Mean citations: {stats['citations_mean']:.1f}")

# Query with filters
recent = cache.query("demo_cache", year_min=2015, limit=5)
print(f"\nRecent papers (2015+): {len(recent)}")

# Cleanup
cache.delete("demo_cache")
print("Cache deleted")

Created cache with 50 papers
Year range: 1959 - 2021
Mean citations: 10035.3

Recent papers (2015+): 5
Cache deleted


## 5. Async Search

Run concurrent searches for better performance.

In [6]:
import asyncio

async def concurrent_search():
    queries = ["machine learning", "deep learning", "neural networks"]
    
    # Run all searches concurrently
    counts = await aio.count_many(queries)
    
    for query, count in counts.items():
        print(f"'{query}': {count:,} matches")

# Run async function
await concurrent_search()

'machine learning': 100 matches
'deep learning': 10 matches
'neural networks': 17 matches


## 6. Enrich IDs

Fetch full metadata for a list of IDs.

In [7]:
# Enrich a list of IDs (resolve full metadata for each).
ids = [w.openalex_id for w in search("neuroscience", limit=2).works]
works = enrich_ids(ids)

for w in works:
    print(f"Title: {w.title}")
    print(f"Year: {w.year}, Citations: {w.cited_by_count:,}")
    print()

Title: The Cognitive neurosciences
Year: 2005, Citations: 4,733

Title: Power failure: why small sample size undermines the reliability of neuroscience
Year: 2013, Citations: 7,865



## CLI Commands

You can also use the CLI:

```bash
# Search
openalex-local search "machine learning" -n 5

# Get by DOI with citation
openalex-local search-by-doi "10.7717/peerj.4375" --citation
openalex-local search-by-doi "10.7717/peerj.4375" --bibtex

# Cache commands
openalex-local cache create mypapers -q "CRISPR" -l 100
openalex-local cache stats mypapers
openalex-local cache export mypapers refs.bib -f bibtex
openalex-local cache delete mypapers --yes
```